# Chuẩn hóa văn bản, công thức toán học và khử trùng lặp dữ liệu

> **Môn học:** Nhập môn Khoa học Dữ liệu — HCMUS 
> **Milestone 2:** Bóc tách Cấu trúc Tài liệu & Chuẩn Hóa Phân Cấp 
> **Các thành viên phụ trách:** Trần Ngọc Thanh Tú (Làm sạch & Chuẩn hóa), Đặng Ngọc Hoàng 
> **Mục tiêu:** Loại bỏ định dạng LaTeX không mang nghĩa, quy chuẩn hóa công thức toán, thực thi cơ chế băm nội dung để khử trùng lặp tự động liên version, và đánh giá trên tập dữ liệu đối sánh gán nhãn thủ công.

---

### Vai trò của chuẩn hóa đối với khử trùng lặp dữ liệu

Đề bài nêu rõ: *"full-text matching should be performed after cleanup to remove discrepancies in minor details such as formatting or spacing that might otherwise cause the code to identify identical texts as different"*.

Nếu hai phiên bản v1 và v2 của một bài báo có cùng một câu nhưng chỉ khác nhau một dấu cách thừa hoặc một lệnh in đậm `\textbf{...}`, việc không chuẩn hóa sẽ khiến hệ thống coi chúng là hai phần tử khác biệt. Khi đó, tỷ lệ khử trùng lặp sẽ sụp đổ. Do đó, **chuẩn hóa là tiền đề quyết định sự thành bại của deduplication**.

In [1]:
import os
import sys
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path("../data") if Path("../data").exists() else Path("data")

def load_or_placeholder(filename: str, desc: str):
    filepath = data_dir / filename
    if not filepath.exists():
        print(f"[Đang xử lý] Chưa có tệp {filename} — {desc}.")
        return None
    df = pd.read_csv(filepath)
    print(f"[Đã sẵn sàng] Đã nạp thành công {filename}: {len(df):,} bản ghi ({desc}).")
    return df

print("______________________________________________________________________________")
print("KHỞI TẠO DỮ LIỆU CHUẨN HÓA & ĐÁNH GIÁ ĐỘ CHÍNH XÁC:")
df_gold = load_or_placeholder("gold_structure.csv", "Tập dữ liệu kiểm định gồm 20 bài báo gán nhãn thủ công (Thanh Tú)")
df_elements = load_or_placeholder("elements_sample.csv", "Mẫu bảng elements sau khử trùng lặp")
print("______________________________________________________________________________")

______________________________________________________________________________
KHỞI TẠO DỮ LIỆU CHUẨN HÓA & ĐÁNH GIÁ ĐỘ CHÍNH XÁC:
[Đang xử lý] Chưa có tệp gold_structure.csv — Tập dữ liệu kiểm định gồm 20 bài báo gán nhãn thủ công (Thanh Tú).
[Đang xử lý] Chưa có tệp elements_sample.csv — Mẫu bảng elements sau khử trùng lặp.
______________________________________________________________________________


### 1. Quy trình chuẩn hóa văn bản và công thức toán học

Module `src/parser/normalize.py` do **Trần Ngọc Thanh Tú** phụ trách thực thi theo hai bước nghiêm ngặt:
1. **`clean_latex(s: str) -> str`:**
 - Xóa bỏ các tùy chọn canh vị trí ảnh/bảng không mang nghĩa ngữ nghĩa (ví dụ: `[htbp!]`).
 - Giữ lại nội dung văn bản bên trong các lệnh nhấn mạnh: `\textbf{text}` $\rightarrow$ `text`, `\textit{text}` $\rightarrow$ `text`, `\emph{text}` $\rightarrow$ `text`.
 - Xóa các lệnh định dạng trống: `\centering`, `\midrule`, `\toprule`, `\bottomrule`, `\hline`, `\noindent`.
 - Thu gọn toàn bộ khoảng trắng thừa và tab về đúng một dấu cách đơn (`[ \t]+` $\rightarrow$ `" "`).
2. **`normalize_math(s: str) -> str`:**
 - **Xử lý khối toán trước:** Chuyển đổi toàn bộ `$$...$$`, `\[...\]`, `\begin{align}`, `\begin{gather}` về dạng thống nhất `\begin{equation}...\end{equation}`.
 - **Xử lý toán nội dòng sau:** Chuyển đổi `\(...\)` về dạng chuẩn `$...$`.
 - *Tính ổn định:* Đảm bảo tính chất toán học $f(f(x)) = f(x)$, chạy nhiều lần không làm biến dạng dữ liệu.

In [2]:
# Thử nghiệm tính ổn định của bộ chuẩn hóa
def demo_clean(s: str) -> str:
    import re
    s = re.sub(r'\\(?:textbf|textit|emph)\{([^}]*)\}', r'\1', s)
    s = re.sub(r'\\(?:centering|midrule|hline)', ' ', s)
    s = re.sub(r'[ \t]+', ' ', s)
    return s.strip()

sample_raw = r"\centering We evaluate \textbf{BERT} with \emph{accuracy} on \textit{ImageNet}."
clean_1 = demo_clean(sample_raw)
clean_2 = demo_clean(clean_1)

print("KIỂM ĐỊNH TÍNH ỔN ĐỊNH CỦA BỘ LÀM SẠCH LATEX:")
print("______________________________________________________________________________")
print(f"Chuỗi gốc        : {sample_raw}")
print(f"Sau lần chạy 1   : {clean_1}")
print(f"Sau lần chạy 2   : {clean_2}")
print(f"Khớp tuyệt đối   : {clean_1 == clean_2} (Thỏa mãn tiêu chuẩn Idempotency)")

KIỂM ĐỊNH TÍNH ỔN ĐỊNH CỦA BỘ LÀM SẠCH LATEX:
______________________________________________________________________________
Chuỗi gốc        : \centering We evaluate \textbf{BERT} with \emph{accuracy} on \textit{ImageNet}.
Sau lần chạy 1   : We evaluate BERT with accuracy on ImageNet.
Sau lần chạy 2   : We evaluate BERT with accuracy on ImageNet.
Khớp tuyệt đối   : True (Thỏa mãn tiêu chuẩn Idempotency)


### 2. Thuật toán sinh định danh phần tử phục vụ khử trùng lặp dữ liệu

Theo hợp đồng dữ liệu `docs/schema_m2.md`, định danh phần tử được sinh bằng hàm băm SHA-1 trên nội dung đã chuẩn hóa:

```text
element_id = f"{pub_key}#{kind_code}_{sha1(kind + ' ' + clean_content)[:8]}"
```

Trong đó cấu trúc định danh gồm ba thành phần:
- `pub_key`: Khóa định danh bài báo (ví dụ `2507-06659`).
- `kind_code`: Ký hiệu rút gọn cho loại phần tử (ví dụ `sec` cho phân mục, `snt` cho câu, `eqn` cho công thức toán).
- `sha1(kind + ' ' + clean_content)[:8]`: 8 ký tự đầu của chuỗi băm SHA-1 trên nội dung đã chuẩn hóa kết hợp loại phần tử.

**Đặc tính kỹ thuật của giải thuật:**
- Khử trùng lặp xảy ra **hoàn toàn tự động**: Nếu hai version có cùng một đoạn văn hoặc công thức toán, hàm băm tự động trả về cùng một `element_id`.
- Tiết kiệm bộ nhớ: Bảng `elements` chỉ lưu đúng 1 dòng cho mỗi nội dung duy nhất; bảng `hierarchy` của các version sau chỉ việc trỏ lại `element_id` đó.

In [3]:
# Mô phỏng trực tiếp cơ chế băm element_uid
def element_uid(pub_key: str, kind: str, clean_content: str) -> str:
    KIND_CODE = {
        "document": "doc", "section": "sec", "subsection": "sub",
        "paragraph": "par", "sentence": "snt", "formula": "eqn", "figure": "fig"
    }
    payload = f"{kind}\x00{clean_content}".encode("utf-8")
    h = hashlib.sha1(payload).hexdigest()[:8]
    return f"{pub_key}#{KIND_CODE.get(kind, 'elt')}_{h}"

pub = "2507-06659"
content_v1 = "Deep neural networks achieve state-of-the-art results on NLP benchmarks."
content_v2 = "Deep neural networks achieve state-of-the-art results on NLP benchmarks." # Giống hoàn toàn
content_v3 = "Deep neural networks achieve competitive results on NLP benchmarks."    # Đã sửa chữ

id_v1 = element_uid(pub, "sentence", content_v1)
id_v2 = element_uid(pub, "sentence", content_v2)
id_v3 = element_uid(pub, "sentence", content_v3)

print("KẾT QUẢ SINH ELEMENT_ID KHỬ TRÙNG LẶP LIÊN PHIÊN BẢN:")
print("______________________________________________________________________________")
print(f"Câu v1 ID : {id_v1}")
print(f"Câu v2 ID : {id_v2} --> Tự động trùng ID với v1: {id_v1 == id_v2} (Dedup thành công)")
print(f"Câu v3 ID : {id_v3} --> Tách ID mới do sửa nội dung: {id_v1 != id_v3}")

KẾT QUẢ SINH ELEMENT_ID KHỬ TRÙNG LẶP LIÊN PHIÊN BẢN:
______________________________________________________________________________
Câu v1 ID : 2507-06659#snt_fb7b9c84
Câu v2 ID : 2507-06659#snt_fb7b9c84 --> Tự động trùng ID với v1: True (Dedup thành công)
Câu v3 ID : 2507-06659#snt_414fb470 --> Tách ID mới do sửa nội dung: True


### 3. Đánh Giá Độ Chính Xác Trên Tập Đối Sánh Gán Nhãn Thủ Công

Hầu hết các nhóm nghiên cứu chỉ dừng lại ở việc báo cáo "tỷ lệ parse thành công", vốn chỉ phản ánh chương trình không bị crash chứ không đo lường được **độ chính xác của cấu trúc được bóc tách**.

Nhóm 04 tiến hành xây dựng **Tập dữ liệu kiểm định gồm 20 bài báo gán nhãn thủ công (`data/gold_structure.csv`)** bao gồm:
- Tên tệp gốc thực sự (`root_file`).
- Số lượng đề mục cấp cao nhất (`n_top_sections`).
- Danh sách tiêu đề các mục đó (`top_section_titles`).

Bằng cách đối sánh trực tiếp cây phân cấp trích xuất tự động với tập dữ liệu đối sánh chuẩn, nhóm đo lường được 3 chỉ số khách quan:
1. **Root Detection Accuracy:** Tỷ lệ tìm đúng file gốc.
2. **Section Hierarchy Accuracy:** Tỷ lệ nhận diện đúng số lượng mục cấp 1.
3. **Title F1-Score:** Độ trùng khớp ngữ nghĩa của danh mục tiêu đề.